<a href="https://colab.research.google.com/github/arthur-cristo/ram-scrap/blob/main/ram_scrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!apt-get update

!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libpango-1.0-0 \
    libcairo2 \
    libasound2 \
    libnss3 \
    libxshmfence1

!pip install playwright

!playwright install chromium

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,695 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libatk-bridge2.0-0 is already the newest version (

In [43]:
import asyncio
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from urllib.parse import urljoin
from google.colab import files

In [4]:
BASE_URL = "https://meupc.net"
LIST_URL = "https://meupc.net/memorias"

resultados = []

In [40]:
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from playwright.async_api import async_playwright

BASE_URL = "https://meupc.net"

MAX_ITENS = 150


def limpar_preco(texto):
    if not texto:
        return None

    texto = (
        texto.replace("R$", "")
        .replace(".", "")
        .replace(",", ".")
        .replace("\xa0", "")
        .strip()
    )

    try:
        return float(texto)
    except:
        return None


async def extrair_memorias_da_tabela(html):

    soup = BeautifulSoup(html, "html.parser")

    tabela = soup.select_one("table tbody")

    if not tabela:
        print("❌ tabela não encontrada")
        return []

    linhas = tabela.select("tr")

    print(f"🔎 linhas encontradas: {len(linhas)}")

    memorias = []

    for i, linha in enumerate(linhas):

        try:

            texto_linha = linha.get_text(" ", strip=True).lower()

            # ignora patrocinado
            if "patrocinado" in texto_linha:
                print(f"⏭️ patrocinado ignorado [{i}]")
                continue

            link = linha.select_one('a[href*="/peca/"]')

            if not link:
                print(f"⚠️ linha sem link [{i}]")
                continue

            nome = link.get_text(" ", strip=True)

            href = link.get("href")

            url = urljoin(BASE_URL, href)

            preco = None
            preco_pix = None

            try:
                preco_td = linha.select_one('td[data-label="Preço"]')

                if preco_td:
                    preco = limpar_preco(
                        preco_td.get_text(" ", strip=True)
                    )

            except Exception as e:
                print("erro preço:", e)

            try:
                preco_pix_td = linha.select_one(
                    'td[data-label="Preço PIX"]'
                )

                if preco_pix_td:
                    preco_pix = limpar_preco(
                        preco_pix_td.get_text(" ", strip=True)
                    )

            except Exception as e:
                print("erro preço pix:", e)

            memoria = {
                "nome": nome,
                "url": url,
                "preco": preco,
                "preco_pix": preco_pix,
            }

            print("\n✅ MEMÓRIA CAPTURADA")
            print(memoria)

            memorias.append(memoria)

        except Exception as e:
            print(f"❌ erro linha [{i}]:", e)

    return memorias


async def main():

    resultados = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage"
            ]
        )

        context = await browser.new_context()

        page = await context.new_page()

        print("🌐 abrindo página inicial...")

        await page.goto(
            "https://meupc.net/memorias",
            wait_until="networkidle"
        )

        api = context.request

        page_num = 0

        while len(resultados) < MAX_ITENS:
            page_num += 1
            print("\n======================")
            print(f"📄 PÁGINA {page_num}")
            print("======================")

            cookies = await context.cookies()
            csrf_token = None
            for cookie in cookies:
                if cookie["name"] == "ci_csrf_cookie":
                    csrf_token = cookie["value"]
                    break
            print("CSRF:", csrf_token)

            response_data = await page.evaluate(
              """
              async ({ pageNum, csrfToken }) => {

                  const formData = new FormData()

                  formData.append("ci_csrf_token", csrfToken)

                  const response = await fetch(
                      `https://meupc.net/api/memorias?comp=true&vel=4,3200&tam=15,16&page=${pageNum}`,
                      {
                          method: "POST",
                          body: formData,
                          credentials: "include"
                      }
                  )

                  return await response.json()
              }
              """,
              {
                  "pageNum": page_num,
                  "csrfToken": csrf_token
              }
            )

            try:
                data = response_data

            except Exception as e:
                print("❌ erro json:", e)
                break

            print("\n🔎 chaves json:")
            print(data.keys())

            pecas = data.get("peca", [])

            if not pecas:
              print("❌ nenhuma peça encontrada")
              break

            for i, peca in enumerate(pecas):

              if len(resultados) >= MAX_ITENS:
                  break

              # ignora patrocinado
              if peca.get("patrocinado") == "1":
                  print(f"⏭️ patrocinado ignorado [{i}]")
                  continue

              nome = peca.get("descricao")

              hash_produto = peca.get("hash")

              slug = peca.get("slug")

              url = f"{BASE_URL}/peca/{hash_produto}/{slug}"

              memoria = {
                  "index": len(resultados) + 1,
                  "nome": nome,
                  "url": url,
                  "preco": peca.get("preco"),
                  "preco_pix": peca.get("preco_boleto"),
                  "preco_por_gb": peca.get("preco_gb"),
                  "modulos": peca.get("modulo"),
                  "capacidade": peca.get("tamanho_padrao"),
                  "hash": hash_produto,
              }

              print("\n✅ MEMÓRIA")
              print(memoria)

              try:

                  historico = await page.evaluate(
                      """
                      async ({ hash, csrfToken }) => {

                          const formData = new FormData()

                          formData.append(
                              "ci_csrf_token",
                              csrfToken
                          )

                          const response = await fetch(
                              `https://meupc.net/api/historico?hash=${hash}&tipo_preco=menor_dia`,
                              {
                                  method: "POST",
                                  body: formData,
                                  credentials: "include"
                              }
                          )

                          return await response.json()
                      }
                      """,
                      {
                          "hash": hash_produto,
                          "csrfToken": csrf_token
                      }
                  )

                  labels = historico.get("labels", [])

                  datasets = historico.get("datasets", [])

                  precos_historico = []

                  if datasets:
                      precos_historico = datasets[0].get("data", [])

                  for data_label, preco_label in zip(
                      labels,
                      precos_historico
                  ):

                      if (
                          data_label is None
                          or preco_label is None
                      ):
                          continue

                      memoria[data_label] = preco_label

                  resultados.append(memoria)

                  print(
                      f"💾 SALVO [{len(resultados)}] "
                      f"{nome}"
                  )

              except Exception as e:

                  print("❌ erro histórico:", e)

    print("\n======================")
    print("📋 RESULTADOS FINAIS")
    print("======================")

    for i, item in enumerate(resultados):

        print(
            f"{i+1}. "
            f"{item['nome']} | "
            f"{item['preco']} | "
            f"{item['preco_pix']}"
        )

    print("\n📊 criando dataframe...")

    df = pd.DataFrame(resultados)

    # colunas fixas
    colunas_base = [
        "index",
        "nome",
        "url",
        "preco",
        "preco_pix",
        "preco_por_gb",
        "modulos",
        "capacidade",
        "hash",
    ]

    # pega colunas de datas
    colunas_datas = [
        c for c in df.columns
        if c not in colunas_base
    ]

    # ordena datas
    colunas_datas = sorted(colunas_datas)

    # reordena dataframe
    df = df[
        [c for c in colunas_base if c in df.columns]
        + colunas_datas
    ]

    print(df.head())

    print("\n📦 total dataframe:", len(df))

    arquivo = "memorias_historico.csv"

    df.to_csv(
        arquivo,
        sep=";",
        encoding="utf-8-sig",
        index=False
    )

    files.download("memorias_historico.csv")

    print("\n✅ arquivo salvo:", arquivo)

In [41]:
await main()

🌐 abrindo página inicial...

📄 PÁGINA 1
CSRF: 47ccb7f35aa80d6dbc68918754d139dd

🔎 chaves json:
dict_keys(['status', 'peca', 'total'])
⏭️ patrocinado ignorado [0]

✅ MEMÓRIA
{'index': 1, 'nome': 'Kingston Fury Beast (Preto)', 'url': 'https://meupc.net/peca/64gcGS/memoria-kingston-fury-beast-kf432c16bbk216', 'preco': '935.00', 'preco_pix': '935.00', 'preco_por_gb': '58.44', 'modulos': '2x8 GB', 'capacidade': '16', 'hash': '64gcGS'}
💾 SALVO [1] Kingston Fury Beast (Preto)

✅ MEMÓRIA
{'index': 2, 'nome': 'Rise Mode Diamond Series (Branco)', 'url': 'https://meupc.net/peca/Yzs6v2/memoria-rise-mode-diamond-series-rm-d4-16g-3200dw', 'preco': '999.90', 'preco_pix': '849.99', 'preco_por_gb': '53.12', 'modulos': '1x16 GB', 'capacidade': '16', 'hash': 'Yzs6v2'}
💾 SALVO [2] Rise Mode Diamond Series (Branco)

✅ MEMÓRIA
{'index': 3, 'nome': 'Rise Mode Z (Preto)', 'url': 'https://meupc.net/peca/bk29mL/memoria-rise-mode-z-rmd416g3200z', 'preco': '1066.61', 'preco_pix': '1066.61', 'preco_por_gb': '66.66